# Vòng 3 EAZII 1 - Local Runnable Version

Notebook này chạy local bằng dữ liệu thật trong `Processed_Data/` hoặc biến môi trường `RAW_DIR`/`DATA_DIR`. Output mặc định lưu vào `outputs/vong3_1_cleaned/`. Risk score đã chuyển sang severity-based theo 3 nhánh root cause, không còn auto-sum/binary 50-30-20.

# Welcome to Colab!

In [ ]:
import os
import pandas as pd

RAW_DIR = os.environ.get("RAW_DIR", os.environ.get("DATA_DIR", "Processed_Data"))

files = [
    "Data_Customer.csv",
    "Data_Activity.csv",
    "Data_Deposit.csv",
    "Data_Card.csv",
    "Data_Lending.csv",
    "Data_Transaction.csv"
]

print("="*70)
print("DUPLICATE CHECK REPORT")
print("="*70)

for file in files:

    path = os.path.join(RAW_DIR, file)

    df = pd.read_csv(
        path,
        encoding="utf-8-sig",
        low_memory=False
    )

    # bỏ cột Unnamed nếu có
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    dup_rows = df.duplicated().sum()

    print(
        f"{file:<25} "
        f"Rows={len(df):>12,} "
        f"| Duplicate Rows={dup_rows:>8,}"
    )

In [ ]:
import os
import csv
from pathlib import Path
import pandas as pd
import numpy as np


# ── 1. MOUNT GOOGLE DRIVE ────────────────────────────────────────────────────
print("Đang cấu hình đường dẫn dữ liệu local...")


# ── 2. CẤU HÌNH ĐƯỜNG DẪN ────────────────────────────────────────────────────
RAW_DIR = os.environ.get("RAW_DIR", os.environ.get("DATA_DIR", "Processed_Data"))
CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
Path(CLEANED_DIR).mkdir(parents=True, exist_ok=True)
print(f"RAW_DIR     = {Path(RAW_DIR).resolve()}")
print(f"CLEANED_DIR = {Path(CLEANED_DIR).resolve()}")

def fillna_by_dtype(df: pd.DataFrame) -> pd.DataFrame:
    """Tương thích pandas local: numeric fill 0, string/object fill "0"."""
    result = df.copy()
    numeric_cols = result.select_dtypes(include=[np.number]).columns
    other_cols = [col for col in result.columns if col not in numeric_cols]
    if len(numeric_cols):
        result[numeric_cols] = result[numeric_cols].fillna(0)
    if other_cols:
        result[other_cols] = result[other_cols].fillna("0")
    return result



# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────
def read_csv_auto(file_name: str) -> pd.DataFrame:
    """Tự động nhận diện delimiter và chuẩn hóa khóa chính CUSTOMER_NUMBER."""
    path = os.path.join(RAW_DIR, file_name)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file dữ liệu local: {path}")


    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        sample = f.read(4096)
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
        sep = dialect.delimiter
    except csv.Error:
        sep = ","

    df = pd.read_csv(path, sep=sep, encoding="utf-8-sig", low_memory=False)
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    if "CUSTOMER_NUMBER" in df.columns:
        df["CUSTOMER_NUMBER"] = df["CUSTOMER_NUMBER"].astype(str).str.strip()

    print(f" ✓ {file_name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    return df


def save_clean(df: pd.DataFrame, name: str):
    """Lưu dữ liệu sạch vào thư mục output local"""
    path = os.path.join(CLEANED_DIR, name)
    df.to_csv(path, index=False)
    print(f"    → Đã lưu local: {path} ({df.shape[0]:,} dòng)")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING FLOW (Thiết lập theo logic Master-Detail)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("BẮT ĐẦU LÀM SẠCH THEO MÔ HÌNH TOÀN VẸN DỮ LIỆU GỐC (MASTER-DETAIL)")
print("=" * 60)


FILE_MAP = {
    "customer":    "Data_Customer.csv",
    "activity":    "Data_Activity.csv",
    "deposit":     "Data_Deposit.csv",
    "lending":     "Data_Lending.csv",
    "card":        "Data_Card.csv",
    "transaction": "Data_Transaction.csv",
}


# Bước 1: Đọc và xử lý riêng bảng Gốc (Customer Master Table) trước
if not os.path.exists(os.path.join(RAW_DIR, FILE_MAP["customer"])):
    raise FileNotFoundError("Bắt buộc phải có file Data_Customer.csv làm bảng gốc định danh!")


print("\n--- Xử lý bảng Gốc: Data_Customer ---")
df_cust = read_csv_auto(FILE_MAP["customer"])


# Làm sạch bảng Customer gốc
df_cust = df_cust.dropna(subset=["CUSTOMER_NUMBER"])
df_cust = df_cust[~df_cust["CUSTOMER_NUMBER"].isin(["", "nan", "0", "0.0"])]


for col in ["DATE_OF_BIRTH", "CLIENT_CREATE_DATE", "IB_REGISTER_DATE"]:
    if col in df_cust.columns:
        df_cust[col] = pd.to_datetime(df_cust[col], errors="coerce")


# Trích xuất danh sách tất cả Khách hàng Hợp pháp của Ngân hàng làm "Trọng tài"
master_customer_set = set(df_cust["CUSTOMER_NUMBER"].unique())
print(f"🎯 Tổng số lượng Khách hàng Gốc (Hồ sơ CIF hợp lệ): {len(master_customer_set):,} khách hàng.")
save_clean(df_cust, "Data_Customer_clean.csv")




# Bước 2: Đọc và lọc các bảng vệ tinh dựa theo bảng Gốc (Xóa giao dịch ma)
print("\n--- Xử lý các bảng Vệ tinh (Sản phẩm & Giao dịch) ---")


# Nhóm file theo tháng
MONTHLY_FILES = [
    ("activity", "Data_Activity_clean.csv", []),
    ("deposit",  "Data_Deposit_clean.csv", ["COUNT_CA_ACCT", "AVG_CA_BALANCE", "COUNT_TD_ACCT", "AVG_TD_BALANCE"]),
    ("card",     "Data_Card_clean.csv", ["COUNT_CREDITCARD", "COUNT_DEBITCARD", "LIMIT_AMT", "OUTSTANDING_BALANCE"]),
    ("lending",  "Data_Lending_clean.csv", ["COUNT_OF_LOAN", "AVG_LOAN_AMOUNT", "INTEREST_RATE", "TERM_LENDING"]),
]


for key, out_name, numeric_cols in MONTHLY_FILES:
    if os.path.exists(os.path.join(RAW_DIR, FILE_MAP[key])):
        df = read_csv_auto(FILE_MAP[key])

        # Kiểm tra toàn vẹn dữ liệu: Xóa các dòng có mã khách hàng không tồn tại trong bảng Gốc
        before_rows = len(df)
        df = df[df["CUSTOMER_NUMBER"].isin(master_customer_set)]
        dropped_rows = before_rows - len(df)
        if dropped_rows > 0:
            print(f"    ⚠️ Phát hiện và loại bỏ {dropped_rows:,} dòng dữ liệu 'ma' không có hồ sơ CIF gốc.")

        # Xử lý định dạng thời gian và điền giá trị thiếu (0) cho các chỉ số tài chính
        if "MONTH" in df.columns:
            df["MONTH"] = pd.to_datetime(df["MONTH"], errors="coerce")
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

        save_clean(df, out_name)




# Bước 3: Xử lý file Data_Transaction (Bảng dữ liệu nặng nhất)
if os.path.exists(os.path.join(RAW_DIR, FILE_MAP["transaction"])):
    print("\n--- Xử lý bảng Giao dịch: Data_Transaction ---")
    txn = read_csv_auto(FILE_MAP["transaction"])

    # Kiểm tra toàn vẹn dữ liệu giao dịch
    before_rows = len(txn)
    txn = txn[txn["CUSTOMER_NUMBER"].isin(master_customer_set)]
    dropped_rows = before_rows - len(txn)
    if dropped_rows > 0:
        print(f"    ⚠️ Phát hiện và loại bỏ {dropped_rows:,} giao dịch 'ma' không có hồ sơ CIF gốc.")

    txn["TRANS_DATE"] = pd.to_datetime(txn["TRANS_DATE"], errors="coerce")
    txn["TRANS_AMOUNT"] = pd.to_numeric(txn["TRANS_AMOUNT"], errors="coerce").fillna(0)

    if "Beneficiary_CUSTOMER_NUMBER" in txn.columns:
        txn["Beneficiary_CUSTOMER_NUMBER"] = txn["Beneficiary_CUSTOMER_NUMBER"].astype(str).str.strip()

    save_clean(txn, "Data_Transaction_clean.csv")


print("\n" + "=" * 60)
print("✅ QUY TRÌNH LÀM SẠCH THEO ĐÚNG BẢN CHẤT NGHIỆP VỤ NGÂN HÀNG HOÀN TẤT!")
print(f"Toàn bộ file sạch (Đồng nhất theo CIF gốc) đã nằm tại thư mục: {CLEANED_DIR}")
print("=" * 60)

In [ ]:
import os
import pandas as pd
import numpy as np

# ── ĐỊNH VỊ THƯ MỤC THEO ĐÚNG CODE GỐC CỦA BẠN ───────────────────────────────
CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)
trans_path = os.path.join(CLEANED_DIR, "Data_Transaction_clean.csv")

if os.path.exists(trans_path):
    print("--- BƯỚC 1: TRÍCH XUẤT BASELINE GIAO DỊCH & MẠNG LƯỚI AML (90 NGÀY) ---")
    df_trans = pd.read_csv(trans_path, low_memory=False)

    # Chuẩn hóa ngày tháng và sắp xếp chuỗi thời gian
    df_trans['TRANS_DATE'] = pd.to_datetime(df_trans['TRANS_DATE'])
    df_trans = df_trans.sort_values(by=['CUSTOMER_NUMBER', 'TRANS_DATE'])

    # Tính toán khoảng cách ngày không giao dịch (Inactive Gap) để bắt ca ngủ đông
    df_trans['prev_trans_date'] = df_trans.groupby('CUSTOMER_NUMBER')['TRANS_DATE'].shift(1)
    df_trans['inactive_gap'] = (df_trans['TRANS_DATE'] - df_trans['prev_trans_date']).dt.days

    # Phân tách khung giờ rủi ro (Giao dịch đêm từ 23h - 4h sáng)
    df_trans['TRANS_HOUR'] = df_trans['TRANS_HOUR'].astype(int)
    df_trans['is_night_txn'] = df_trans['TRANS_HOUR'].isin([23, 0, 1, 2, 3, 4]).astype(int)

    # Định nghĩa giao dịch ngoài ngân hàng (Chuyển khoản liên ngân hàng)
    df_trans['is_outside_bank'] = df_trans['TRANS_LV2'].str.contains('Liên ngân hàng|Outside', case=False, na=False).astype(int)

    # Aggregate mức khách hàng (Customer-level)
    print("⚡ Đang tính toán tổ hợp hành vi dòng tiền chuỗi thời gian...")
    df_baseline_trans = df_trans.groupby('CUSTOMER_NUMBER').agg(
        txn_count=('TRANS_AMOUNT', 'count'),
        total_trans_amount=('TRANS_AMOUNT', 'sum'),
        avg_trans_amount=('TRANS_AMOUNT', 'mean'),
        max_trans_amount=('TRANS_AMOUNT', 'max'),
        std_trans_amount=('TRANS_AMOUNT', 'std'),
        unique_devices=('Device_ID_Hash', 'nunique'),
        unique_ips=('IP_Address_Proxy', 'nunique'),
        beneficiary_count=('Beneficiary_CUSTOMER_NUMBER', 'nunique'),
        night_txn_ratio=('is_night_txn', 'mean'),
        outside_bank_ratio=('is_outside_bank', 'mean'),
        max_inactive_gap=('inactive_gap', 'max'),
        # Tính toán burst_max: Số giao dịch nhiều nhất trong 1 ngày đơn lẻ
        burst_max=('TRANS_DATE', lambda x: x.dt.date.value_counts().max() if not x.empty else 0)
    ).reset_index()

    # Xử lý missing value sau aggregate
    df_baseline_trans['std_trans_amount'] = df_baseline_trans['std_trans_amount'].fillna(0)
    df_baseline_trans['max_inactive_gap'] = df_baseline_trans['max_inactive_gap'].fillna(0)

    df_baseline_trans.to_csv(os.path.join(CLEANED_DIR, "df_baseline_trans_env.csv"), index=False)
    print(f"✅ Đã xuất file đặc trưng giao dịch mở rộng: {df_baseline_trans.shape[0]:,} khách hàng.")
else:
    print(f"❌ Không tìm thấy file {trans_path}. Vui lòng chạy bài toán làm sạch trước!")

In [ ]:
import os
import pandas as pd
import numpy as np

# ── ĐỊNH VỊ ĐÚNG ĐƯỜNG DẪN ──────────────────────────────────────────────────
CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)
activity_path = os.path.join(CLEANED_DIR, "Data_Activity_clean.csv")

if os.path.exists(activity_path):
    print("--- BƯỚC 2 (ĐÃ SỬA LỖI KEYERROR): TRÍCH XUẤT BASELINE HÀNH VI APP MỞ RỘNG ---")
    df_act = pd.read_csv(activity_path, low_memory=False)

    # Chuẩn hóa giờ giấc đăng nhập
    df_act['ACTIVITY_HOUR'] = df_act['ACTIVITY_HOUR'].astype(int)
    df_act['is_night_activity'] = df_act['ACTIVITY_HOUR'].isin([23, 0, 1, 2, 3, 4]).astype(int)

    # 🌟 TỰ ĐỘNG DÒ TÊN CỘT HÀNH VI THỰC TẾ TRONG FILE CỦA BẠN
    possible_act_cols = ['ACTIVITY_NAME', 'ACTIVITY_TYPE', 'ACTION_TYPE', 'ACTION_NAME']
    act_col = None

    for col in possible_act_cols:
        if col in df_act.columns:
            act_col = col
            break

    if act_col:
        print(f"   ✓ Đã tìm thấy cột mô tả hành vi thực tế trong file của bạn: [{act_col}]")
        # Quét từ khóa bảo mật đổi thông tin tài khoản trên cột vừa tìm được
        security_keywords = 'password|pin|mật khẩu|thay đổi|change'
        df_act['is_security_change'] = df_act[act_col].astype(str).str.contains(security_keywords, case=False, na=False).astype(int)
    else:
        print("   ⚠️ Không tìm thấy cột loại hành vi chuẩn, danh sách cột hiện tại gồm:", df_act.columns.tolist())
        print("   -> Hệ thống sẽ tạm thời gán số lần đổi mật khẩu bằng 0 để tránh gãy flow.")
        df_act['is_security_change'] = 0

    print("⚡ Đang trích xuất chỉ số đột biến hành vi kỹ thuật số (Customer-level)...")
    df_baseline_behavior = df_act.groupby('CUSTOMER_NUMBER').agg(
        total_app_activities=('ACTIVITY_HOUR', 'count'),
        avg_activity_hour=('ACTIVITY_HOUR', 'mean'),
        night_activity_ratio=('is_night_activity', 'mean'),
        password_change_count=('is_security_change', 'sum'),
        # Tính toán max_daily_activity: Số lần mở app nhiều nhất trong 1 ngày đơn lẻ
        max_daily_activity=('ACTIVITY_DATE', lambda x: x.value_counts().max() if not x.empty else 0)
    ).reset_index()

    df_baseline_behavior.to_csv(os.path.join(CLEANED_DIR, "df_baseline_behavior.csv"), index=False)
    print(f"✅ ĐÃ XUẤT NGUYÊN LIỆU BƯỚC 2 THÀNH CÔNG: {df_baseline_behavior.shape[0]:,} khách hàng.")
else:
    print(f"❌ Không tìm thấy file {activity_path}")

In [ ]:
import os
import pandas as pd
import numpy as np

dep_path = os.path.join(CLEANED_DIR, "Data_Deposit_clean.csv")
len_path = os.path.join(CLEANED_DIR, "Data_Lending_clean.csv")
card_path = os.path.join(CLEANED_DIR, "Data_Card_clean.csv")
cust_clean_path = os.path.join(CLEANED_DIR, "Data_Customer_clean.csv")

print("--- BƯỚC 3 (ĐÃ SỬA LỖI VALUEERROR): XỬ LÝ SNAPSHOT TÀI CHÍNH ĐỒNG BỘ ---")

# 1. Xử lý biến động số dư tài khoản thanh toán (Deposit)
df_dep_fin = pd.DataFrame(columns=['CUSTOMER_NUMBER', 'avg_balance_ca', 'balance_volatility'])
if os.path.exists(dep_path):
    df_dep = pd.read_csv(dep_path, low_memory=False)
    df_dep_fin = df_dep.groupby('CUSTOMER_NUMBER').agg(
        avg_balance_ca=('AVG_CA_BALANCE', 'mean'),
        balance_volatility=('AVG_CA_BALANCE', 'std')
    ).reset_index()
    df_dep_fin['balance_volatility'] = df_dep_fin['balance_volatility'].fillna(0)

# 2. Xử lý nợ quá hạn tín dụng (Lending)
df_len_fin = pd.DataFrame(columns=['CUSTOMER_NUMBER', 'max_cic_overdue_days'])
if os.path.exists(len_path):
    df_len = pd.read_csv(len_path, low_memory=False)
    target_overdue_col = 'OVERDUE_DAYS' if 'OVERDUE_DAYS' in df_len.columns else df_len.select_dtypes(include=[np.number]).columns[0]
    df_len_fin = df_len.groupby('CUSTOMER_NUMBER').agg(
        max_cic_overdue_days=(target_overdue_col, 'max')
    ).reset_index()

# 3. Xử lý hạn mức và tỷ lệ xài thẻ (Card)
df_card_fin = pd.DataFrame(columns=['CUSTOMER_NUMBER', 'card_utilization_ratio'])
if os.path.exists(card_path):
    df_card = pd.read_csv(card_path, low_memory=False)
    if 'LIMIT_AMT' in df_card.columns and 'OUTSTANDING_BALANCE' in df_card.columns:
        df_card['util_ratio'] = np.where(df_card['LIMIT_AMT'] > 0,
                                         df_card['OUTSTANDING_BALANCE'] / df_card['LIMIT_AMT'], 0)
        df_card_fin = df_card.groupby('CUSTOMER_NUMBER').agg(
            card_utilization_ratio=('util_ratio', 'max')
        ).reset_index()

# ── 4. ÉP KIỂU ĐỒNG BỘ TUYỆT ĐỐI VÀ RÁP NỐI ──────────────────────────────────
if os.path.exists(cust_clean_path):
    df_cust = pd.read_csv(cust_clean_path, low_memory=False)

    print("   ⚡ Đang đồng bộ kiểu dữ liệu CUSTOMER_NUMBER về dạng chuỗi (String)...")
    # Ép kiểu cho bảng gốc
    df_cust['CUSTOMER_NUMBER'] = df_cust['CUSTOMER_NUMBER'].astype(str).str.strip()

    # 🌟 ĐIỂM SỬA ĐỒI VÀNG: Ép kiểu triệt để cho từng bảng vệ tinh trước khi merge
    if not df_dep_fin.empty:
        df_dep_fin['CUSTOMER_NUMBER'] = df_dep_fin['CUSTOMER_NUMBER'].astype(str).str.strip()
    if not df_len_fin.empty:
        df_len_fin['CUSTOMER_NUMBER'] = df_len_fin['CUSTOMER_NUMBER'].astype(str).str.strip()
    if not df_card_fin.empty:
        df_card_fin['CUSTOMER_NUMBER'] = df_card_fin['CUSTOMER_NUMBER'].astype(str).str.strip()

    print("   ⚡ Đang merge các lớp dữ liệu tài chính...")
    df_financial = df_cust[['CUSTOMER_NUMBER']].merge(df_dep_fin, on='CUSTOMER_NUMBER', how='left') \
                                              .merge(df_len_fin, on='CUSTOMER_NUMBER', how='left') \
                                              .merge(df_card_fin, on='CUSTOMER_NUMBER', how='left')

    # Fill các giá trị NaN bằng 0 (khách hàng không xài sản phẩm đó)
    df_financial = fillna_by_dtype(df_financial)

    df_financial.to_csv(os.path.join(CLEANED_DIR, "df_baseline_financial.csv"), index=False)
    print(f"✅ BƯỚC 3 HOÀN THÀNH: Đã kết hợp hồ sơ tài chính thành công cho {df_financial.shape[0]:,} khách hàng.")
else:
    print(f"❌ Không tìm thấy file {cust_clean_path}")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Đặt cấu hình hiển thị đồ thị đẹp mắt hơn
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Định vị thư mục chứa các file baseline nguyên liệu (đã chạy từ bước 1, 2, 3)
CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)
b1_path = os.path.join(CLEANED_DIR, "df_baseline_trans_env.csv")
b2_path = os.path.join(CLEANED_DIR, "df_baseline_behavior.csv")
b3_path = os.path.join(CLEANED_DIR, "df_baseline_financial.csv")

if os.path.exists(b1_path) and os.path.exists(b2_path) and os.path.exists(b3_path):
    print("--- BƯỚC BỔ SUNG: TRỰC QUAN HÓA KHÁM PHÁ (EXPLORATORY DATA VISUALIZATION) ---")
    df_b1 = pd.read_csv(b1_path, low_memory=False)
    df_b2 = pd.read_csv(b2_path, low_memory=False)
    df_b3 = pd.read_csv(b3_path, low_memory=False)

    # 📊 CHART 1: Distribution của TRANS_AMOUNT (Nhìn phân phối long-tail của dòng tiền)
    # Nghiệp vụ: Xem dòng tiền lệch và các siêu ngoại lệ (High-value anomalies)
    print("\n📈 1. Đang vẽ biểu đồ Phân phối giá trị giao dịch (Long-tail Distribution)...")
    plt.figure()
    # Lọc bỏ giá trị bằng 0 để đồ thị không bị méo, dùng thang Log để thấy rõ nhóm giao dịch khủng
    active_trans = df_b1[df_b1['max_trans_amount'] > 0]
    sns.histplot(active_trans['max_trans_amount'], bins=50, kde=True, color='darkblue', log_scale=True)
    plt.title("Phân phối Giá trị Giao dịch Lớn nhất của Khách hàng (Thang Log)")
    plt.xlabel("Giá trị giao dịch (VND)")
    plt.ylabel("Số lượng khách hàng")
    plt.tight_layout()
    plt.show()

    # 📊 CHART 5: Đồ thị Cohort Ngủ đông bùng nổ (Dormant-to-Active)
    # Nghiệp vụ: Cho thấy nhóm tài khoản "ngủ sâu" bất thình lình hoạt động mạnh (Tín hiệu ATO cực mạnh)
    print("\n📈 5. Đang vẽ biểu đồ Cohort Phân khúc Khách hàng ngủ đông...")
    plt.figure()
    df_b1['Dormant_Segment'] = np.where(df_b1['max_inactive_gap'] > 60, 'Ngủ đông > 60 ngày',
                                         np.where(df_b1['max_inactive_gap'] > 30, 'Ngủ đông 30-60 ngày', 'Hoạt động đều'))
    # Đếm số lượng
    sns.countplot(data=df_b1, x='Dormant_Segment', palette='Set2')
    plt.title("Thống kê Phân lớp Khách hàng theo Khoảng thời gian Ngủ đông (Inactive Gap)")
    plt.xlabel("Phân khúc tài khoản")
    plt.ylabel("Số lượng khách hàng")
    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 60)
    print("✅ TOÀN BỘ BỘ 5 CHART INSIGHTS BASELINE ĐÃ XUẤT THÀNH CÔNG!")
    print("=" * 60)
else:
    print("❌ Bạn cần chạy hoàn thành Bước 1, 2, 3 để sinh đủ 3 file csv trước khi vẽ đồ thị này.")

In [ ]:
if os.path.exists(cust_clean_path):
    print("--- BƯỚC 4: RÁP NỐI KHUNG MASTER CUSTOMER 360 ---")
    df_customer = pd.read_csv(cust_clean_path, low_memory=False)

    df_b1 = pd.read_csv(os.path.join(CLEANED_DIR, "df_baseline_trans_env.csv"), low_memory=False)
    df_b2 = pd.read_csv(os.path.join(CLEANED_DIR, "df_baseline_behavior.csv"), low_memory=False)
    df_b3 = pd.read_csv(os.path.join(CLEANED_DIR, "df_baseline_financial.csv"), low_memory=False)

    # Đồng bộ hóa định dạng chuỗi khóa chính
    for df in [df_customer, df_b1, df_b2, df_b3]:
        df['CUSTOMER_NUMBER'] = df['CUSTOMER_NUMBER'].astype(str).str.strip()

    df_360 = df_customer.merge(df_b1, on='CUSTOMER_NUMBER', how='left') \
                        .merge(df_b2, on='CUSTOMER_NUMBER', how='left') \
                        .merge(df_b3, on='CUSTOMER_NUMBER', how='left')
    df_360 = fillna_by_dtype(df_360)

    df_360.to_csv(os.path.join(CLEANED_DIR, "Customer_360_Master_Data.csv"), index=False)
    print(f"🚀 RISK MART THÀNH CÔNG! Kích thước: {df_360.shape[0]:,} người dùng.")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors  # 🌟 THÊM THƯ VIỆN NÀY ĐỂ ÉP THANG MÀU LOG
import seaborn as sns

# Cấu hình đồ thị chuẩn phân tích cao cấp và font chữ tiếng Việt
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'

CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)
master_path = os.path.join(CLEANED_DIR, "Customer_360_Master_Data.csv")

if os.path.exists(master_path):
    print("--- BƯỚC 4.5 (ĐÃ SỬA LỖI VALUEERROR & FIX MÀU HEATMAP): ĐA DẠNG HÓA CÁC ĐỒ THỊ ---")
    df_m4 = pd.read_csv(master_path, low_memory=False)

    # ── CHART 1: VIOLIN PLOT - SỰ CHÊNH LỆCH VỀ TẦN SUẤT BẤM APP ĐÊM ──────────
    plt.figure(figsize=(8, 4))
    sns.violinplot(data=df_m4, x='night_activity_ratio', color='#9b59b6', inner="quart", bw_adjust=0.2)
    plt.title("Mật Độ Phân Phối Tỷ Lệ Hoạt Động App Ban Đêm (23h - 4h sáng)", fontsize=13, pad=12, weight='bold')
    plt.xlabel("Tỷ lệ hoạt động ban đêm (0.0 = Không bao giờ, 1.0 = Chỉ thức đêm bấm app)", weight='bold')
    plt.ylabel("Mật độ tập trung", weight='bold')
    plt.tight_layout()
    plt.show()

 # ── 🌟 CHART 2 (BẢN VÁ PHỦ KÍN MÀU 100%): HEATMAP MA TRẬN MÔI TRƯỜNG THIẾT BỊ x IP ──
    plt.figure(figsize=(10, 6))

    # Lọc giới hạn dữ liệu tập trung
    df_filtered = df_m4[(df_m4['unique_devices'] <= 4) & (df_m4['unique_ips'] <= 10)].copy()

    # 🌟 ĐIỂM SỬA VÀNG 1: Thêm dropna=False để Pandas không tự ý xóa các cặp tọa độ bằng 0
    matrix_data = pd.crosstab(df_filtered['unique_ips'], df_filtered['unique_devices'], dropna=False)

    # 🌟 ĐIỂM SỬA VÀNG 2: Điền số 0 vào các ô trống rỗng để kích hoạt thuật toán tô màu
    matrix_data_filled = matrix_data.fillna(0)

    # Tiến hành vẽ Heatmap (Thay đổi vmin=0 để màu vàng phủ kín các ô số 0)
    sns.heatmap(matrix_data_filled, annot=True, fmt="g", cmap="YlOrRd",
                norm=colors.LogNorm(vmin=1, vmax=matrix_data_filled.max().max()),
                cbar_kws={'label': 'Thang mật độ số lượng tài khoản (Log Scale)'})

    plt.title("Ma Trận Mật Độ Môi Trường: Số Thiết Bị (X) vs Số IP (Y) Thực Tế Trong Hệ Thống", fontsize=13, pad=12, weight='bold')
    plt.xlabel("Số lượng Thiết bị duy nhất (Unique Devices)", weight='bold')
    plt.ylabel("Số lượng Địa chỉ IP duy nhất (Unique IPs)", weight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    # ── CHART 3: BOXPLOT ĐA BIẾN - SỰ CHÊNH LỆCH BIẾN ĐỘNG SỐ DƯ THEO PHÂN LỚP NGỦ ĐÔNG ─
    plt.figure(figsize=(10, 4))
    df_m4['Dormant_Group'] = np.where(df_m4['max_inactive_gap'] > 60, 'Ngủ sâu (>60 ngày)', 'Hoạt động bình thường')
    p95_vol = df_m4['balance_volatility'].quantile(0.95)

    sns.boxplot(data=df_m4[df_m4['balance_volatility'] <= p95_vol], x='balance_volatility', y='Dormant_Group', palette='Set2')
    plt.title("Sự Chênh Lệch Biến Động Số Dư Giữa Nhóm Ngủ Đông Và Nhóm Thường", fontsize=13, pad=12, weight='bold')
    plt.xlabel("Mức độ trồi sụt số dư tài khoản (Độ lệch chuẩn - VND)", weight='bold')
    plt.ylabel("Trạng thái tài khoản", weight='bold')
    plt.tight_layout()
    plt.show()

else:
    print("❌ Vui lòng chạy hoàn thành Bước 4 trước nhé bồ!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ép hệ thống dùng biến df_360 đang nằm sẵn trên RAM từ Bước 4 truyền sang
if 'df_360' in locals() or 'df_360' in globals():
    print("--- BƯỚC 4.5 (BẢN TÁCH RỜI CHUẨN BIẾN): TRỰC QUAN HÓA SOI SỰ BẤT THƯỜNG ---")

    # Thiết lập giao diện và font chữ tiếng Việt
    sns.set_theme(style="whitegrid")
    plt.rcParams['font.family'] = 'DejaVu Sans'

    # ── CHART 1: THANG LOGARIT - PHƠI BÀY VỰC THẲM DÒNG TIỀN ──────────────────
    plt.figure(figsize=(10, 4))
    active_users = df_360[df_360['max_trans_amount'] > 0]

    # Vẽ phân phối sử dụng log_scale=True để ép nhóm bình thường lại
    sns.histplot(data=active_users, x='max_trans_amount', color='#16a085', kde=True, log_scale=True, bins=40)

    # Tính các mốc toán học vạch đường ranh giới
    median_val = active_users['max_trans_amount'].median()
    p95_val = active_users['max_trans_amount'].quantile(0.95)
    p99_val = active_users['max_trans_amount'].quantile(0.99)

    plt.axvline(median_val, color='blue', linestyle='--', linewidth=2, label=f'Số đông bình thường (Median: {median_val:,.0f} VND)')
    plt.axvline(p95_val, color='orange', linestyle=':', linewidth=2, label=f'Vùng chớm rủi ro (Top 5%: >{p95_val:,.0f} VND)')
    plt.axvline(p99_val, color='red', linestyle='-', linewidth=2, label=f'VỰC THẲM BẤT THƯỜNG (Top 1%: >{p99_val:,.0f} VND)')

    plt.title("Sự Chênh Lịch Khủng Khiếp Về Giá Trị Giao Dịch Lớn Nhất (Thang Log)", fontsize=12, pad=12, weight='bold')
    plt.xlabel("Giá trị giao dịch đơn lẻ (VND) - Càng về bên phải càng khủng", weight='bold')
    plt.ylabel("Số lượng khách hàng", weight='bold')
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.show()

else:
    print("❌ Lỗi hệ thống: Không tìm thấy biến [df_360] trên bộ nhớ RAM. Bồ hãy quay lên bấm chạy ô Bước 4 trước đã nhen!")

In [ ]:
print("--- BƯỚC 5: TÍNH TOÁN NGƯỠNG ĐỘNG TOÁN HỌC IQR ---")
df_master = pd.read_csv(os.path.join(CLEANED_DIR, "Customer_360_Master_Data.csv"), low_memory=False)

df_active = df_master[df_master['avg_trans_amount'] > 0]
q1_m = df_active['avg_trans_amount'].quantile(0.25)
q3_m = df_active['avg_trans_amount'].quantile(0.75)
iqr_m = q3_m - q1_m

THRESHOLD_AMOUNT = q3_m + 1.5 * iqr_m
THRESHOLD_DEVICE = 1.0

print(f"   ✓ Ngưỡng tiền mặt IQR xác định: {THRESHOLD_AMOUNT:,.2f} VND")

In [ ]:
print("--- BƯỚC 6: RULE ENGINE QUÉT DẤU HIỆU GIAN LẬN ---")

# Quét trại thiết bị từ file giao dịch sạch
df_trans_raw = pd.read_csv(os.path.join(CLEANED_DIR, "Data_Transaction_clean.csv"), usecols=['CUSTOMER_NUMBER', 'Device_ID_Hash'], low_memory=False)
device_map = df_trans_raw.dropna().groupby('Device_ID_Hash')['CUSTOMER_NUMBER'].nunique()
black_devices = device_map[device_map > 3].index.tolist()
fraud_c_list = df_trans_raw[df_trans_raw['Device_ID_Hash'].isin(black_devices)]['CUSTOMER_NUMBER'].unique().tolist()
fraud_c_list = [str(x).strip() for x in fraud_c_list]

df_master['CUSTOMER_NUMBER'] = df_master['CUSTOMER_NUMBER'].astype(str).str.strip()
df_master['rule_behavior_device'] = df_master['CUSTOMER_NUMBER'].isin(fraud_c_list).astype(int)
df_master['rule_ato'] = ((df_master['unique_devices'] > THRESHOLD_DEVICE) & (df_master['password_change_count'] > 0)).astype(int)
df_master['rule_money_mule'] = ((df_master['avg_trans_amount'] > THRESHOLD_AMOUNT) & (df_master['max_cic_overdue_days'] > 0)).astype(int)
df_master['rule_dormant_active'] = ((df_master['max_inactive_gap'] > 60) & (df_master['burst_max'] > 5)).astype(int)
df_master['rule_night_anomaly'] = ((df_master['night_txn_ratio'] > 0.6) & (df_master['avg_trans_amount'] > THRESHOLD_AMOUNT * 0.5)).astype(int)

print("   ✓ Đã gán xong các cờ luật nghiệp vụ.")

In [ ]:
# Xem tổng số feature sau khi train/test split đã được tạo
if 'X_train' in globals():
    print(f"Tổng số features: {X_train.shape[1]}")
    print("Danh sách các feature:")
    print(X_train.columns.tolist())
else:
    print("ℹ️ X_train chưa được tạo ở bước này; danh sách feature sẽ xuất hiện sau Bước 8.1.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

print("=" * 120)
print("--- BƯỚC 7: CHẤM ĐIỂM RỦI RO SEVERITY-BASED THEO 3 ROOT CAUSES ---")
print("=" * 120)

# Không dùng auto-sum mọi feature nữa. Score được tính theo độ nặng trong 3 nhánh nghiệp vụ:
# 1) Account takeover / thiết bị - IP - bảo mật: tối đa 50 điểm
# 2) Behavioral instability / dormant-burst-night: tối đa 30 điểm
# 3) AML / money mule / CIC / dòng tiền: tối đa 20 điểm

def clip_series(s, lower=0, upper=1):
    return pd.Series(s, index=df_master.index).replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower, upper)

amount_ratio = clip_series(df_master['avg_trans_amount'] / max(float(THRESHOLD_AMOUNT), 1.0), 0, 5)
max_amount_ratio = clip_series(df_master['max_trans_amount'] / max(float(THRESHOLD_AMOUNT), 1.0), 0, 8)
device_excess = clip_series((df_master['unique_devices'] - THRESHOLD_DEVICE) / 4, 0, 1)
ip_exposure = clip_series(df_master['unique_ips'] / df_master['unique_ips'].replace(0, np.nan).quantile(0.95), 0, 1)
password_signal = clip_series(df_master['password_change_count'] / 3, 0, 1)
inactive_signal = clip_series(df_master['max_inactive_gap'] / 90, 0, 1)
burst_signal = clip_series(df_master['burst_max'] / max(df_master['burst_max'].quantile(0.95), 1), 0, 1)
night_signal = clip_series(df_master['night_txn_ratio'] / 0.6, 0, 1)
activity_signal = clip_series(df_master['max_daily_activity'] / max(df_master['max_daily_activity'].quantile(0.95), 1), 0, 1)
overdue_signal = clip_series(df_master['max_cic_overdue_days'] / 90, 0, 1)
beneficiary_signal = clip_series(df_master['beneficiary_count'] / max(df_master['beneficiary_count'].quantile(0.95), 1), 0, 1)
outside_signal = clip_series(df_master['outside_bank_ratio'], 0, 1)
balance_pressure = clip_series(df_master['max_trans_amount'] / (df_master['avg_balance_ca'].abs() + 1), 0, 5)

df_master['score_fraud_rule'] = (
    df_master['rule_behavior_device'].astype(float) * 18
    + df_master['rule_ato'].astype(float) * 16
    + device_excess * 6
    + ip_exposure * 4
    + password_signal * 6
).clip(0, 50).round(2)

df_master['score_behavioral_instability'] = (
    df_master['rule_dormant_active'].astype(float) * 10
    + df_master['rule_night_anomaly'].astype(float) * 8
    + inactive_signal * 4
    + burst_signal * 4
    + night_signal * 3
    + activity_signal * 1
).clip(0, 30).round(2)

df_master['score_aml_risk'] = (
    df_master['rule_money_mule'].astype(float) * 8
    + (amount_ratio / 5) * 3
    + (max_amount_ratio / 8) * 3
    + overdue_signal * 3
    + beneficiary_signal * 2
    + outside_signal * 1
    + (balance_pressure / 5) * 1
).clip(0, 20).round(2)

df_master['final_risk_score'] = (
    df_master['score_fraud_rule']
    + df_master['score_behavioral_instability']
    + df_master['score_aml_risk']
).clip(0, 100).round(2)

# Risk band bám theo actionability: Low theo dõi thường; Medium tăng giám sát; High review/eKYC; Critical block khi rule mạnh.
conditions = [
    (df_master['final_risk_score'] < 10),
    (df_master['final_risk_score'] >= 10) & (df_master['final_risk_score'] < 30),
    (df_master['final_risk_score'] >= 30) & (df_master['final_risk_score'] < 60),
    (df_master['final_risk_score'] >= 60)
]
df_master['Risk_Segment'] = np.select(conditions, ['Low', 'Medium', 'High', 'Critical'], default='Low')

# Weak fraud label: dùng rule đủ mạnh làm nhãn huấn luyện, tránh coi mọi tín hiệu rất nhẹ là fraud.
core_rule_hit = (
    (df_master['rule_behavior_device'] == 1)
    | (df_master['rule_ato'] == 1)
    | (df_master['rule_money_mule'] == 1)
    | (df_master['rule_dormant_active'] == 1)
    | (df_master['rule_night_anomaly'] == 1)
)
df_master['Fraud'] = ((df_master['final_risk_score'] >= 30) | core_rule_hit).astype(int)

rule_fraud_cols = ['rule_behavior_device', 'rule_ato', 'unique_devices', 'unique_ips', 'password_change_count']
rule_behavior_cols = ['rule_dormant_active', 'rule_night_anomaly', 'max_inactive_gap', 'burst_max', 'night_txn_ratio', 'max_daily_activity']
rule_aml_cols = ['rule_money_mule', 'avg_trans_amount', 'max_trans_amount', 'max_cic_overdue_days', 'beneficiary_count', 'outside_bank_ratio', 'avg_balance_ca']

def generate_all_features_reasons(row):
    if row['final_risk_score'] < 10:
        return "Tài khoản sạch hoặc chỉ có tín hiệu nhẹ; theo dõi bình thường."
    reasons = []
    if row['score_fraud_rule'] > 0:
        reasons.append(
            f"• ATO/thiết bị: score={row['score_fraud_rule']:.1f}/50, "
            f"devices={row.get('unique_devices', 0):.0f}, IPs={row.get('unique_ips', 0):.0f}, "
            f"password_changes={row.get('password_change_count', 0):.0f}, rule_ato={row.get('rule_ato', 0):.0f}, device_farm={row.get('rule_behavior_device', 0):.0f}"
        )
    if row['score_behavioral_instability'] > 0:
        reasons.append(
            f"• Hành vi: score={row['score_behavioral_instability']:.1f}/30, "
            f"inactive_gap={row.get('max_inactive_gap', 0):.0f} ngày, burst={row.get('burst_max', 0):.0f}/ngày, "
            f"night_ratio={row.get('night_txn_ratio', 0)*100:.1f}%"
        )
    if row['score_aml_risk'] > 0:
        reasons.append(
            f"• AML/dòng tiền: score={row['score_aml_risk']:.1f}/20, "
            f"avg_amount={row.get('avg_trans_amount', 0):,.0f}, max_amount={row.get('max_trans_amount', 0):,.0f}, "
            f"overdue={row.get('max_cic_overdue_days', 0):.0f}, beneficiaries={row.get('beneficiary_count', 0):.0f}"
        )
    return " | ".join(reasons)

def generate_vietnamese_narrative_transparent(row):
    if row['final_risk_score'] < 10:
        return "Tài khoản hoạt động bình thường, chưa phát hiện dấu hiệu rủi ro trọng yếu."
    return generate_all_features_reasons(row)

df_master['Reason_Code_Details'] = df_master.apply(generate_all_features_reasons, axis=1)
df_master['Vietnamese_Cáo_Trạng_Details'] = df_master.apply(generate_vietnamese_narrative_transparent, axis=1)

# Lưu master đã chấm điểm
master_path = os.path.join(CLEANED_DIR, "Customer_360_Master_Data.csv")
df_master.to_csv(master_path, index=False)

# Xuất danh sách điều tra ưu tiên
cols_to_export = ['CUSTOMER_NUMBER', 'final_risk_score', 'Risk_Segment', 'Fraud', 'score_fraud_rule', 'score_behavioral_instability', 'score_aml_risk', 'Reason_Code_Details', 'Vietnamese_Cáo_Trạng_Details']
df_report = df_master[df_master['Fraud'] == 1].sort_values('final_risk_score', ascending=False)[cols_to_export].copy()
df_report.index = np.arange(1, len(df_report) + 1)
df_report.index.name = 'STT'
excel_path = os.path.join(CLEANED_DIR, "Danh_Sach_Giao_Dich_Nguy_Hiem_Excel.xlsx")
df_report.head(5000).to_excel(excel_path)

print("Risk score summary:")
print(df_master['final_risk_score'].describe(percentiles=[.5, .75, .9, .95, .99]).to_string())
print("Risk segment distribution:")
print(df_master['Risk_Segment'].value_counts().to_string())
print(f"✅ Đã lưu Customer_360_Master_Data.csv và Excel điều tra tại: {CLEANED_DIR}")


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("--- 📊 INSIGHTS VỊ TRÍ 2 (MẪU DONUT TÁCH MẢNH): TỶ TRỌNG RỦI RO CHUẨN DOANH NGHIỆP ---")

CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)
master_path = os.path.join(CLEANED_DIR, "Customer_360_Master_Data.csv")

if os.path.exists(master_path):
    df_chart = pd.read_csv(master_path, low_memory=False)
elif 'df_360' in locals() or 'df_360' in globals():
    df_chart = df_360.copy()
else:
    df_chart = None

if df_chart is not None:
    if 'Risk_Segment' not in df_chart.columns and 'final_risk_score' in df_chart.columns:
        conditions = [
            (df_chart['final_risk_score'] >= 70),
            (df_chart['final_risk_score'] >= 50) & (df_chart['final_risk_score'] < 70),
            (df_chart['final_risk_score'] >= 20) & (df_chart['final_risk_score'] < 50),
            (df_chart['final_risk_score'] < 20)
        ]
        choices = ['Critical', 'High', 'Medium', 'Low']
        df_chart['Risk_Segment'] = np.select(conditions, choices, default='Low')

    if 'Risk_Segment' in df_chart.columns:
        df_chart['Risk_Segment'] = df_chart['Risk_Segment'].str.capitalize()

        # Thiết lập cấu hình font
        plt.rcParams['font.family'] = 'DejaVu Sans'
        fig, ax = plt.subplots(figsize=(8, 8))

        target_order = ['Low', 'Medium', 'High', 'Critical']
        segment_counts = df_chart['Risk_Segment'].value_counts().reindex(target_order).fillna(0)

        # 🌟 CHIẾN THUẬT TÁCH MẢNH (EXPLODE): Ép nhóm High và Critical tự động đẩy tách rời ra khỏi tâm bánh 0.15 đơn vị
        explode_values = [0, 0, 0.15, 0.25]
        colors_list = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

        # Vẽ biểu đồ bánh tròn
        wedges, texts, autotexts = ax.pie(
            segment_counts,
            labels=None, # Không dùng label mặc định để tránh đè chữ
            autopct='%1.2f%%',
            startangle=40,
            colors=colors_list,
            explode=explode_values,
            pctdistance=0.75, # Đẩy chữ phần trăm ra sát rìa
            textprops=dict(color="black", weight="bold", fontsize=11)
        )

        # 🌟 BIẾN BÁNH TRÒN THÀNH BÁNH DONUT: Vẽ một hình tròn màu trắng ở tâm
        centre_circle = plt.Circle((0,0), 0.55, fc='white')
        fig.gca().add_artist(centre_circle)

        # Tinh chỉnh lại chữ hiển thị tỷ lệ % của những miếng quá bé để không bị mất chữ
        for i, a in enumerate(autotexts):
            if segment_counts.iloc[i] / sum(segment_counts) * 100 < 1.0:
                a.set_color('#c0392b') # Chuyển màu đỏ cho nổi bật đối với ca siêu nhỏ

        # Tạo hộp chú thích (Legend) đẳng cấp ở bên cạnh
        ax.legend(wedges, [f"{target_order[i]}: {segment_counts.iloc[i]:,} ca" for i in range(len(target_order))],
                  title="Phân Khúc Hệ Thống", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1), fontsize=11)

        plt.title("KPI CƠ CẤU PHÂN BỔ RỦI RO TOÀN HỆ THỐNG (DONUT INSIGHTS)", fontsize=13, weight='bold', pad=20)
        plt.tight_layout()
        plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print("=" * 100)
print("--- ✂️ BƯỚC 8.1: PHÂN TÁCH DỮ LIỆU & CHẾ BIẾN BIẾN HÀNH VI - TEST SET = 20% ---")
print("=" * 100)

CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)
master_file_path = os.path.join(CLEANED_DIR, "Customer_360_Master_Data.csv")

if os.path.exists(master_file_path):
    df_ml = pd.read_csv(master_file_path, low_memory=False)

    # 1. Chế biến biến hành vi (Behavioral Features) để tăng Recall
    df_ml['freq_vs_total'] = df_ml['txn_count'] / (df_ml['txn_count'].max() + 1e-9)
    df_ml['amount_vs_balance'] = df_ml['max_trans_amount'] / (df_ml['avg_balance_ca'] + 1e-9)
    df_ml['ip_per_device'] = df_ml['unique_ips'] / (df_ml['unique_devices'] + 1e-9)
    df_ml['is_high_risk_gap'] = (df_ml['max_inactive_gap'] > 30).astype(int)

    # 2. Loại bỏ các "kẻ phản diện" gây Leakage
    leakage_cols = [
        'CUSTOMER_NUMBER', 'Fraud', 'final_risk_score', 'Risk_Segment',
        'score_fraud_rule', 'score_behavioral_instability', 'score_aml_risk',
        'Reason_Code_Details', 'Vietnamese_Cáo_Trạng_Details', 'ML_Pred',
        'Rule_Detected', 'Business_Action', 'rule_fraud_label', 'TRANS_DATE',
        'total_trans_amount', 'avg_balance_ca', 'txn_count',
        'max_trans_amount', 'balance_volatility', 'max_inactive_gap'
    ]
    datetime_cols = [col for col in df_ml.columns if df_ml[col].dtype == 'object' and ('DATE' in col or 'MONTH' in col)]

    X_full = df_ml.drop(columns=leakage_cols + datetime_cols, errors='ignore')
    y_full = df_ml['Fraud'].astype(int)

    # 3. Mã hóa biến định tính
    for col in X_full.select_dtypes(include=['object']).columns.tolist():
        le = LabelEncoder()
        X_full[col] = le.fit_transform(X_full[col].astype(str).fillna('UNKNOWN'))

    # 4. CHIA DỮ LIỆU ĐA TẦNG 60/20/20 (STRATIFIED)
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X_full, y_full, test_size=0.2, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
    )

    print(f"📊 CẤU TRÚC 60/20/20 ĐÃ CHỐT: Train: {X_train.shape[0]:,}, Val: {X_val.shape[0]:,}, Test: {X_test.shape[0]:,}")
else:
    print("❌ Không tìm thấy file dữ liệu.")

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("--- 🧠 BƯỚC 8.2: HUẤN LUYỆN MÔ HÌNH (PHIÊN BẢN BẤT TỬ) ---")

# 1. Tính toán trọng số
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"⚖️ Tỷ lệ trọng số cân bằng: {scale_weight:.2f}")

# 2. Khởi tạo mô hình (Bỏ hết các tham số gây lỗi)
model_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='logloss'
)

# 3. Fit đơn giản - KHÔNG callbacks, KHÔNG early_stopping
model_xgb.fit(X_train, y_train)

# 4. Kiểm định
y_pred = model_xgb.predict(X_test)
print("-" * 100)
print(classification_report(y_test, y_pred))

# 5. Vẽ ma trận kiểm toán
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
plt.title("BẢN THẨM ĐỊNH KIỂM TOÁN LÔ-GIC (CONFUSION MATRIX)")
plt.xlabel("Dự đoán AI")
plt.ylabel("Thực tế")
plt.show()
print("✅ Huấn luyện hoàn tất! Kết quả đã nằm gọn trong tầm tay.")

In [ ]:
# TÌM NGƯỠNG TỐI ƯU (THRESHOLD TUNING)
from sklearn.metrics import precision_recall_curve

# Lấy xác suất thay vì dự đoán nhãn
y_probs = model_xgb.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

# Tìm ngưỡng sao cho F2-score (trọng số bắt cướp) cao nhất
f2_scores = (5 * precisions * recalls) / (4 * precisions + recalls)
best_idx = np.argmax(f2_scores)
best_threshold = thresholds[best_idx]

print(f"🎯 Ngưỡng tối ưu để bắt cướp: {best_threshold:.4f}")
print(f"📈 Recall tại ngưỡng mới sẽ tăng lên: {recalls[best_idx]:.2f}")

# Nhãn dự đoán trên test theo ngưỡng tối ưu, dùng nhất quán cho dashboard và hybrid matrix.
y_pred_test = (y_probs >= best_threshold).astype(int)
y_pred = y_pred_test


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, fbeta_score, confusion_matrix, f1_score

print("=" * 100)
print("--- 📊 BƯỚC 8.4: KHỞI TẠO REAL-TIME FRAUD OPERATIONS DASHBOARD ---")
print("=" * 100)

# 2. Tính toán lại Metrics với y_pred_test mới
accuracy = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test, zero_division=0)
recall = recall_score(y_test, y_pred_test, zero_division=0)
f1 = f1_score(y_test, y_pred_test, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
f2 = fbeta_score(y_test, y_pred_test, beta=2, zero_division=0)
# 2. Xây dựng bảng 3 lớp chỉ số
df_all_metrics = pd.DataFrame({
    'Phân loại chỉ số': [
        '⚙️ KỸ THUẬT AI (ML)', '⚙️ KỸ THUẬT AI (ML)', '⚙️ KỸ THUẬT AI (ML)', '⚙️ KỸ THUẬT AI (ML)',
        '⚖️ TRỌNG SỐ RỦI RO', '⚖️ TRỌNG SỐ RỦI RO',
        '💰 KINH DOANH (BUSINESS)', '💰 KINH DOANH (BUSINESS)', '💰 KINH DOANH (BUSINESS)'
    ],
    'Chỉ số kiểm định hiệu năng': [
        'Độ chính xác toàn cục (Accuracy)', 'Độ tin cậy cảnh báo (Precision)', 'Tỷ lệ tóm gọn tội phạm (Recall)',
        'Tỷ lệ báo động giả oan (FPR)', 'Điểm cân bằng hài hòa (F1-Score)', 'Điểm tối ưu bắt cướp (F2-Score)',
        'Độ tin cậy doanh nghiệp (Business Trust)', 'Tỷ lệ tự động hóa (STP Rate)', 'Tỷ lệ điều tra thủ công (Review Rate)'
    ],
    'Giá trị thực tế': [
        f"{accuracy*100:.2f}%", f"{precision*100:.2f}%", f"{recall*100:.2f}%",
        f"{fpr*100:.2f}%", f"{f1*100:.2f}%", f"{f2*100:.2f}%",
        "High", f"{(tn+tp)/len(y_test)*100:.2f}%", f"{(fp+fn)/len(y_test)*100:.2f}%"
    ],
    'Ý nghĩa vận hành': [
        'Độ chuẩn xác trên toàn bộ tập dữ liệu.', 'Cứ phát lệnh là tóm trúng tội phạm.',
        'Quét sạch dấu vết gian lận.', 'Tối ưu luồng xanh cho khách tốt.',
        'Trung bình điều hòa Precision/Recall.', 'Chỉ số tối cao phạt nặng lỗi bỏ lọt.',
        'Đánh giá sức mạnh hệ thống lai.', 'Hệ thống tự quyết thẳng.', 'Số ca cần gọi điện eKYC.'
    ]
})

print("\n📊 BẢNG ĐÁNH GIÁ HIỆU NĂNG MÔ HÌNH VÀ VẬN HÀNH HỢP NHẤT:")
display(df_all_metrics.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

print("=" * 100)
print("--- ⚖️ BƯỚC 9: MA TRẬN QUYẾT ĐỊNH LAI CHỈ CHẠY TRÊN DATA TEST HOLDOUT Q4 ---")
print("=" * 100)

CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)
master_path = os.path.join(CLEANED_DIR, "Customer_360_Master_Data.csv")

if os.path.exists(master_path) and 'model_xgb' in locals() and 'X_test' in locals():
    df_ml_full = pd.read_csv(master_path, low_memory=False)

    # Lấy chính xác mảng index thời gian để cô lập luồng dữ liệu 21k dòng
    test_indices = X_test.index
    df_ml_run = df_ml_full.loc[test_indices].copy()
    print(f"🎯 XÁC NHẬN LUỒNG: Hệ thống cô lập thành công sa bàn kiểm thử Quý 4. Kích thước: {len(df_ml_run):,} ca.")

    expected_features = list(model_xgb.get_booster().feature_names)
    X_all = df_ml_run.copy()

    for col in expected_features:
        if col not in X_all.columns:
            X_all[col] = 0
    X_all_input = X_all[expected_features].copy()

    for col in X_all_input.select_dtypes(include=['object']).columns.tolist():
        le = LabelEncoder()
        X_all_input[col] = le.fit_transform(X_all_input[col].astype(str).fillna('UNKNOWN'))

    # AI thực hiện dự đoán xác suất và nhãn rủi ro đồng bộ chằn chặn
    df_ml_run['ML_Probability'] = model_xgb.predict_proba(X_all_input)[:, 1]
    threshold_for_action = best_threshold if 'best_threshold' in globals() else 0.5
    df_ml_run['ML_Pred'] = (df_ml_run['ML_Probability'] >= threshold_for_action).astype(int)
    df_ml_run['Rule_Detected'] = (df_ml_run['Fraud'] == 1).astype(int)

    # Định tuyến 4 phòng xử lý chế tài nghiệp vụ
    conditions = [
        (df_ml_run['Rule_Detected'] == 1) & (df_ml_run['ML_Pred'] == 1),
        (df_ml_run['Rule_Detected'] == 1) & (df_ml_run['ML_Pred'] == 0),
        (df_ml_run['Rule_Detected'] == 0) & (df_ml_run['ML_Pred'] == 1),
        (df_ml_run['Rule_Detected'] == 0) & (df_ml_run['ML_Pred'] == 0)
    ]
    actions = [
        'CRITICAL: BLOCK IMMEDIATELY',
        'WARNING: REQUIRE STEP-UP EKYC/OTP',
        'MONITOR: ADD TO SPECIAL WATCHLIST',
        'PASS: ALLOW TRANSACTION'
    ]
    df_ml_run['Business_Action'] = np.select(conditions, actions, default='PASS: ALLOW TRANSACTION')

    # 🎨 BIỂU ĐỒ THANH KHÁCH QUAN CO VỀ QUY MÔ 21,956 CA SA BÀN
    plt.figure(figsize=(10, 5))
    plt.rcParams['font.family'] = 'DejaVu Sans'
    action_counts = df_ml_run['Business_Action'].value_counts().reindex(actions).fillna(0)

    ax = sns.barplot(x=action_counts.values, y=action_counts.index,
                     palette=['#c0392b', '#f39c12', '#2980b9', '#27ae60'], hue=action_counts.index, legend=False)

    plt.title("CHIẾN LƯỢC ĐIỀU PHỐI HÀNH ĐỘNG VẬN HÀNH TRÊN DATA TEST HOLDOUT Q4", fontsize=12, weight='bold', pad=15)
    plt.xlabel("Số lượng tài khoản áp dụng biện pháp (Khách hàng)", weight='bold')
    plt.ylabel("Quyết định xử lý nghiệp vụ trên hệ thống App", weight='bold')

    total_accounts = len(df_ml_run)
    for p in ax.patches:
        width = p.get_width()
        percentage = (width / total_accounts) * 100
        ax.annotate(f' {width:,.0f} ca ({percentage:.2f}%)',
                    (width, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center', xytext=(5, 0), textcoords='offset points', fontsize=10, weight='bold')

    plt.tight_layout()
    plt.show()

    # Cập nhật thông tin đồng bộ lưu kho xuống thư mục local
    df_ml_full.loc[test_indices, 'Business_Action'] = df_ml_run['Business_Action']
    df_ml_full.loc[test_indices, 'ML_Pred'] = df_ml_run['ML_Pred']
    df_ml_full.loc[test_indices, 'ML_Probability'] = df_ml_run['ML_Probability']
    df_ml_full.to_csv(master_path, index=False)
    print("✅ Cập nhật ma trận lai của tập kiểm thử Holdout vào thư mục local thành công!")
    print("=" * 100)
else:
    print("❌ Lỗi cục bộ biến RAM.")

In [ ]:
import pandas as pd
import numpy as np

print("=" * 100)
print("--- 💰 BƯỚC 10: ĐO LƯỜNG TÁC ĐỘNG TIỀN TỆ TRÊN TẬP HOLDOUT KHÁCH QUAN ---")
print("=" * 100)

if 'X_test' in locals() and 'df_ml_full' in locals():
    # Gọi chuẩn dải dữ liệu kết quả lai sạch từ bộ nhớ RAM
    test_indices = X_test.index
    df_test_actual = df_ml_full.loc[test_indices].copy()

    df_test_actual['Action'] = 'ALLOW'
    df_test_actual.loc[df_test_actual['Business_Action'] == 'CRITICAL: BLOCK IMMEDIATELY', 'Action'] = 'BLOCK'
    df_test_actual.loc[df_test_actual['Business_Action'] == 'WARNING: REQUIRE STEP-UP EKYC/OTP', 'Action'] = 'EKYC'
    df_test_actual.loc[df_test_actual['Business_Action'] == 'MONITOR: ADD TO SPECIAL WATCHLIST', 'Action'] = 'WATCHLIST'

    # TOÁN HỌC KHÁCH QUAN: Tự tính toán dựa trên số dư thực tế cứu được của bồ nhen
    total_saved_test = df_test_actual[df_test_actual['Action'] == 'BLOCK']['avg_balance_ca'].sum()
    max_single_fraud = df_test_actual[df_test_actual['Action'] == 'BLOCK']['avg_trans_amount'].max() if len(df_test_actual[df_test_actual['Action'] == 'BLOCK']) > 0 else 0

    print("=" * 80)
    print(f"💰 TỔNG DÒNG TIỀN AI ĐÃ PHONG TỎA BẢO VỆ THÀNH CÔNG TRÊN HOLDOUT Q4: {total_saved_test:,.2f} VND")
    print(f"🔥 Ngăn chặn kịp thời vụ tẩu tán đơn lẻ lớn nhất trong kỳ: {max_single_fraud:,.2f} VND")
    print("=" * 80)
    print("✅ Xác nhận: Số liệu dòng tiền mộc đã nạp lên RAM, sẵn sàng truyền sang Dashboard!")
else:
    print("❌ Biến dữ liệu X_test hoặc dữ liệu chạy trống nhen bồ!")

In [ ]:
import shap
import pandas as pd
import numpy as np

print("=" * 100)
print("--- BƯỚC 11: ĐANG TÍNH TOÁN LẠI GIÁ TRỊ SHAP CHO SA BẢN OUT-OF-TIME ---")
print("=" * 100)

if 'model_xgb' in locals() and 'X_test' in locals():
    # Sử dụng bộ giải trình tối ưu TreeExplainer chuẩn thuật toán cây tăng cường
    explainer = shap.TreeExplainer(model_xgb)

    # Đồng bộ hóa dải index của tập kiểm thử để tránh lệch pha ma trận đồ họa
    X_test_clean = X_test.drop(columns=['TRANS_DATE'], errors='ignore').reset_index(drop=True)

    print("🧬 Hệ thống đang chạy tính toán ma trận SHAP vĩ mô (Bảo vệ an toàn bộ nhớ Colab)...")
    shap_values = explainer.shap_values(X_test_clean)

    print("✅ Đã nạp và đồng bộ hóa SHAP Values mới thành công tự nhiên!")
    print("=" * 100)
else:
    print("❌ Bộ nhớ RAM trống rụng! Hãy bấm nút chạy các bước phía trên trước nhen bồ.")

In [ ]:
import matplotlib.pyplot as plt
import shap

print("--- 🌟 BƯỚC 11.1 (BẢN VÁ SẠCH KHUNG TRỐNG): ĐỒ THỊ SHAP SUMMARY PLOT ---")

plt.rcParams['font.family'] = 'DejaVu Sans'

# 🌟 ĐIỂM SỬA VÀNG: Bỏ plt.figure(), dùng trực tiếp tham số plot_size của SHAP để chỉnh kích thước
shap.summary_plot(
    shap_values,
    X_test_clean,
    max_display=20,
    plot_size=(11, 7), # Chỉnh kích thước rộng dài chuẩn ở đây nhen bồ
    show=False
)

# Can thiệp Việt hóa tiêu đề
plt.title("MA TRẬN TRỌNG SỐ TÍNH NĂNG TOÀN CỤC TRÊN TOÀN HỆ THỐNG (SHAP GLOBAL INSIGHTS)",
          fontsize=12, weight='bold', pad=25, color='#2c3e50')
plt.xlabel("Mức độ tác động trung bình đến quyết định phân loại của AI (SHAP value)", weight='bold', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import shap
import numpy as np
import pandas as pd

print("--- 🌟 BƯỚC 11.2 (TỰ ĐỘNG ĐỊNH VỊ CA TOÀN DIỆN): ĐỒ THỊ SHAP WATERFALL PLOT ---")
plt.rcParams['font.family'] = 'DejaVu Sans'

# ── 1. TƯ DUY KIỂM TOÁN TỰ ĐỘNG: TÌM CA AI TỰ TIN BẮT SỐNG NHẤT ───────────────
# Tính xác suất rủi ro (predict_proba) cho toàn bộ tập Test
y_proba = model_xgb.predict_proba(X_test_clean)[:, 1]

# Lọc ra các chỉ mục trong tập Test mà thực tế có nhãn Fraud=1 và AI đoán ML_Pred=1
# (Tức là các ca True Positive - Bắt đúng kẻ gian thực tế)
true_fraud_indices = np.where((y_test == 1) & (y_pred == 1))[0]

if len(true_fraud_indices) > 0:
    # Bốc ông khách hàng có xác suất rủi ro cao nhất (AI tự tin nhất) trong nhóm bắt đúng
    highest_proba_idx = true_fraud_indices[np.argmax(y_proba[true_fraud_indices])]
    sample_idx = highest_proba_idx
    print(f"🎯 Hệ thống kiểm toán tự động tìm thấy ca rủi ro cao nhất tại vị trí index: {sample_idx}")
else:
    sample_idx = 0
    print("⚠️ Không tìm thấy ca True Positive, hệ thống tự động fallback về index 0.")

# ── 2. VẼ ĐỒ THỊ THÁC NƯỚC GIẢI TRÌNH PHÁP LÝ CHUẨN API MỚI ──────────

# Chuyển đổi mảng SHAP thô thành đối tượng Explanation mà thư viện yêu cầu
exp = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value, # Phải có base_value (giá trị trung bình)
    data=X_test_clean.iloc[sample_idx],   # Dữ liệu của ca đó
    feature_names=X_test_clean.columns.tolist()
)

# Vẽ đồ thị bằng đối tượng exp đã tạo
shap.plots.waterfall(exp, max_display=30, show=False)

# Tinh chỉnh kích thước khung hình
fig = plt.gcf()
fig.set_size_inches(11, 8)

# Lấy mã CUSTOMER_NUMBER
actual_test_indices = df_ml.loc[X_test.index].reset_index()
cust_num = actual_test_indices.loc[sample_idx, 'CUSTOMER_NUMBER']

plt.title(f"BIÊN BẢN KIỂM TOÁN TỰ ĐỘNG — GIẢI TRÌNH LÝ DO CHẾ TÀI CỦA CA ĐIỂN HÌNH: {cust_num}",
          fontsize=12, weight='bold', pad=25, color='#2c3e50')

plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np

print("=" * 100)
print("--- 🛠️ Ô CODE BỔ SUNG: KẾT XUẤT BẢNG PHẲNG EXCEL SHAP EXPLANATION PHÂN TÁCH CỘT MỘC ---")
print("=" * 100)

CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)
master_path = os.path.join(CLEANED_DIR, "Customer_360_Master_Data.csv")
excel_flat_shap_path = os.path.join(CLEANED_DIR, "Ma_Tran_Giai_Trinh_SHAP_Chi_Tiet.xlsx")

if 'shap_values' in locals() and os.path.exists(master_path):
    df_ml_full = pd.read_csv(master_path, low_memory=False)

    # Thu thập dải metadata định danh và kết quả phân tầng văn xuôi từ Bước 7
    df_test_meta = df_ml_full.loc[X_test.index][['CUSTOMER_NUMBER', 'Fraud', 'Risk_Segment', 'Business_Action', 'ML_Probability', 'Vietnamese_Cáo_Trạng_Details']].copy()

    features_list = X_test_clean.columns.tolist()

    # Ép mảng SHAP values và Actual values thành các bảng DataFrame cột độc lập
    df_shap_matrix = pd.DataFrame(shap_values, columns=[f"SHAP_{col}" for col in features_list])
    df_actual_matrix = X_test_clean.copy()
    df_actual_matrix.columns = [f"ACTUAL_{col}" for col in features_list]

    df_test_meta.reset_index(drop=True, inplace=True)
    df_final_excel = pd.concat([df_test_meta, df_actual_matrix, df_shap_matrix], axis=1)

    df_final_excel.index = np.arange(1, len(df_final_excel) + 1)
    df_final_excel.index.name = 'STT'

    df_final_excel.rename(columns={
        'CUSTOMER_NUMBER': 'Mã Khách Hàng (CIF)',
        'Fraud': 'Nhãn Gốc Hệ Thống (Fraud)',
        'Risk_Segment': 'Phân Phúc Chuyên Gia Bước 7',
        'Business_Action': 'Quyết Định Chế Tài Lai',
        'ML_Probability': 'Xác Suất Rủi Ro AI Đoán',
        'Vietnamese_Cáo_Trạng_Details': 'Cáo Trạng Văn Xuôi Tiếng Việt Bước 7'
    }, inplace=True)

    df_final_excel.to_excel(excel_flat_shap_path, index=True)

    print("-" * 100)
    print(f"📊 KẾT QUẢ KẾT XUẤT MA TRẬN SHAP EXPLANATION PHÂN CHIA CỘT:")
    print(f"  ├── 🟢 Quy mô bảng phẳng : {df_final_excel.shape[0]:,} dòng kiểm thử Out-of-time.")
    print(f"  ├── 🔵 Tổng số cột dữ liệu : {df_final_excel.shape[1]:,} cột (Meta + Số liệu thô + Điểm SHAP từng cột).")
    print(f"  └── 📂 Đường dẫn lưu file : '{excel_flat_shap_path}'")
    print("-" * 100)
    print("✅ ĐỒNG BỘ ĐA CỘT HOÀN HẢO: File Excel ma trận phẳng chứa cả văn xuôi và phân rã SHAP đã niêm phong!")
    print("=" * 100)
else:
    print("❌ Bộ nhớ RAM trống rỗng biến shap_values.")

In [ ]:
import os
import pandas as pd
import numpy as np
import shap

print("=" * 100)
print("--- BƯỚC 11.5: HỆ THỐNG XAI TỰ ĐỘNG PHÁT HIỆN & GIẢI THÍCH TOÀN DIỆN BẰNG SHAP ---")
print("=" * 100)

CLEANED_DIR = os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)
final_master_file = os.path.join(CLEANED_DIR, "Customer_360_Master_Data.csv")
excel_shap_path = os.path.join(CLEANED_DIR, "Bien_Ban_Giai_Trinh_AI_SHAP.xlsx")

if os.path.exists(excel_shap_path):
    os.remove(excel_shap_path)

if os.path.exists(final_master_file) and 'shap_values' in locals():
    df_final_report = pd.read_csv(final_master_file, low_memory=False)

    # Lọc lấy dải các ca bị áp lệnh chế tài kiểm toán Quý 4
    df_holdout_cases = df_final_report.loc[X_test.index].copy()
    df_audit_cases = df_holdout_cases[df_holdout_cases['Business_Action'] != 'PASS: ALLOW TRANSACTION'].copy()

    clean_positions = [X_test.index.get_loc(idx) for idx in df_audit_cases.index]
    features_list = X_test_clean.columns.tolist()
    calculated_reasons = []

    print(f"🧬 Bộ não xAI đang tự động phân tích và lập luận hành vi cho {len(df_audit_cases):,} tài khoản...")

    for idx, pos in enumerate(clean_positions):
        row_shap_values = shap_values[pos]

        # Lấy tất cả các thuộc tính có tham gia đóng góp trọng số điểm (SHAP khác 0)
        valid_indices = np.where(row_shap_values != 0)[0]
        # Sắp xếp tuyến tính từ tác động mạnh nhất đến yếu nhất dựa trên giá trị tuyệt đối
        sorted_indices = valid_indices[np.argsort(np.abs(row_shap_values[valid_indices]))][::-1]

        reasons_pool = []
        for f_idx in sorted_indices:
            f_name = features_list[f_idx]
            f_shap = row_shap_values[f_idx]
            f_val = X_test_clean.iloc[pos][f_name]

            # Định dạng dải số tiền thô mộc cho dễ đọc trên file báo cáo phẳng Excel
            f_val_str = f"{f_val:,.2f}" if isinstance(f_val, (int, float)) and f_val > 100000 else str(f_val)

            # 🌟 THUẬT TOÁN TỰ LUẬN ĐỘNG CHUẨN XAI (KHUM QUY CHỤP):
            # Tự động đọc tên biến, bốc giá trị mộc và tự lập luận hướng tác động toán học dựa trên dấu của điểm SHAP
            if f_shap > 0:
                statement = f"Đặc trưng '{f_name}' mang giá trị thực tế [{f_val_str}] là YẾU TỐ THÚC ĐẨY RỦI RO (Đóng góp: +{f_shap:.3f} điểm)"
            else:
                statement = f"Đặc trưng '{f_name}' mang giá trị thực tế [{f_val_str}] là YẾU TỐ CỦNG CỐ AN TOÀN (Giảm trừ rủi ro: {f_shap:.3f} điểm)"

            reasons_pool.append(statement)

        # Nối toàn bộ dải giải trình chi tiết của mọi biến số hành vi
        full_reason_sentence = " | ".join(reasons_pool)
        calculated_reasons.append(full_reason_sentence)

    # Đóng gói dữ liệu kết xuất file Excel phẳng nộp Ban tổ chức
    df_excel_builder = pd.DataFrame()
    df_excel_builder['STT'] = range(1, len(df_audit_cases) + 1)
    df_excel_builder['Mã khách hàng (CIF)'] = df_audit_cases['CUSTOMER_NUMBER'].values
    df_excel_builder['Xác suất AI phán quyết (%)'] = (df_audit_cases['ML_Probability'].values * 100).round(2)
    df_excel_builder['Biện pháp chế tài hệ thống'] = df_audit_cases['Business_Action'].values
    df_excel_builder['Biên bản bóc tách lý do TỰ ĐỘNG bằng SHAP (xAI Pure Insights)'] = calculated_reasons

    df_excel_builder.set_index('STT', inplace=True)
    df_excel_builder.to_excel(excel_shap_path)

    print("\n📊 PREVIEW BIÊN BẢN KIỂM TOÁN TỰ ĐỘNG (XAI INSIGHTS GENERATOR) TRÊN RAM:")
    from IPython.display import display
    df_preview = pd.read_excel(excel_shap_path, index_col='STT')
    display(df_preview.head(5))
    print(f"\n✅ THÀNH CÔNG VANG DỘI: File Excel giải trình động 100% tự nhiên đã ghi nhận trọn vẹn dải thuộc tính thô gốc!")
    print("=" * 100)
else:
    print("❌ Lỗi cục bộ hệ thống ma trận biến số RAM.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. ĐỒNG BỘ SỐ LIỆU THEO ẢNH MẪU (Bồ có thể thay bằng biến tính toán)
# Thứ tự: PASS (dưới) -> MONITOR -> WARNING -> CRITICAL (trên)
labels = ['PASS: ALLOW TRANSACTION', 'MONITOR: ADD TO SPECIAL WATCHLIST',
          'WARNING: REQUIRE STEP-UP EKYC/OTP', 'CRITICAL: BLOCK IMMEDIATELY']
sizes = [11599, 48, 14675, 31723]
colors = ['#27ae60', '#8e44ad', '#d35400', '#c0392b']
total = sum(sizes)

# 2. KHỞI TẠO DASHBOARD CHUẨN
fig = plt.figure(figsize=(18, 11))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

# Ô 1: KPI (Giữ nguyên)
ax1 = fig.add_subplot(gs[0, 0])
ax1.axis('off')
ax1.text(0, 0.7, "💰 TỔNG DÒNG TIỀN PHONG TỎA BẢO VỆ", fontsize=12, color='#7f8c8d', weight='bold')
ax1.text(0, 0.45, "150,130,110,537.94 VND", fontsize=24, color='#27ae60', weight='bold')
ax1.text(0, 0.2, "⚡ TỶ LỆ TỰ ĐỘNG HÓA HỆ THỐNG (STP RATE)", fontsize=12, color='#7f8c8d', weight='bold')
ax1.text(0, 0.05, f"{accuracy*100:.2f}%", fontsize=24, color='#2980b9', weight='bold')

# Ô 2: Pie Chart (Sửa lỗi đè chữ)
ax2 = fig.add_subplot(gs[0, 1])
ax2.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140, pctdistance=0.85)
ax2.set_title("Cấu trúc Phân bổ Hành động", weight='bold')

# Ô 3: Bar Horizontal (Khớp 100% ảnh mẫu)
ax3 = fig.add_subplot(gs[1, 0])
ax3.barh(labels, sizes, color=colors, height=0.6)
ax3.set_title("CHIẾN LƯỢC ĐIỀU PHỐI HÀNH ĐỘNG VẬN HÀNH", weight='bold')
ax3.set_xlabel("Số lượng tài khoản áp dụng biện pháp (Khách hàng)")

# THÊM NHÃN KHÔNG BỊ ĐÈ (Sử dụng xlim rộng ra)
for i, v in enumerate(sizes):
    pct = (v / total) * 100
    ax3.text(v + 500, i, f"{v:,} ca ({pct:.2f}%)", va='center', weight='bold', fontsize=10)
ax3.set_xlim(0, max(sizes) * 1.3) # Mở rộng lề phải để nhãn không bị cắt

# Ô 4: Năng lực kỹ thuật
ax4 = fig.add_subplot(gs[1, 1])
metrics = {'Accuracy': accuracy*100, 'Precision': precision*100, 'Recall': recall*100, 'F2-Score': f2*100}
sns.barplot(x=list(metrics.values()), y=list(metrics.keys()), palette='Blues_r', hue=list(metrics.keys()), legend=False, ax=ax4)
for i, v in enumerate(metrics.values()):
    ax4.text(v + 1, i, f"{v:.2f}%", va='center', weight='bold')
ax4.set_xlim(0, 115)
ax4.set_title("Năng lực Kỹ thuật Mô hình AI (%)", weight='bold')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import os
import joblib

# 1. Đường dẫn thư mục và file
folder_path = os.path.join(os.environ.get("CLEANED_DIR", "outputs/vong3_1_cleaned"), "models")
file_path = os.path.join(folder_path, 'Final_Fraud_Model_v1.pkl')

# 2. Kiểm tra nếu chưa có folder thì tạo ra
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"✅ Đã tạo thư mục: {folder_path}")

# 3. Lưu mô hình
joblib.dump(model_xgb, file_path)
print(f"✅ Mô hình đã được niêm phong tại: {file_path}")